# Lobo Python smoke test

This notebook verifies a small resting book, user-liquidity summaries, storage displays, and the live Polars lazy-level adapter. It is intentionally compact and deterministic.

In [ ]:
from uuid import NAMESPACE_URL, uuid5

from lobo import Book
from lobo.levels import levels_lazy
from lobo.orders import LimitOrder, Side

TRADER = uuid5(NAMESPACE_URL, "lobo-notebook-trader")
book = Book()

## Add and inspect resting liquidity

A non-crossing sell limit order rests on the ask side. The assertions make this cell double as an executable smoke test.

In [ ]:
first_result = book.fill(LimitOrder(100, 100, TRADER, Side.Sell))
assert first_result.remaining_order_id is not None
assert book.best_ask == 100
assert book.orders.asks.order_count == 1
book.orders

In [ ]:
summary = book.get_user_order_summary(TRADER)
assert summary.order_count == 1
assert summary.visible_quantity == 100
assert summary.visible_value == 10_000
summary

## Verify the live lazy-level view

The lazy adapter reads current levels only when collected. Reusing the same lazy frame after another order is added should expose the updated book.

In [ ]:
asks = book.orders.asks
lazy_asks = levels_lazy(asks)
before = lazy_asks.collect()

assert before.height == 1
assert before["quantity"].sum() == 100
before

In [ ]:
second_result = book.fill(LimitOrder(101, 50, TRADER, Side.Sell))
assert second_result.remaining_order_id is not None

after = lazy_asks.collect()
assert after.height == 2
assert after["quantity"].sum() == 150
after

## Result

The Python API, rich storage objects, user summary, and live lazy-level adapter all completed successfully. Extend this smoke test with a crossing order when validating matching behavior.